# TFM - Data Integration

## Objective

The purpose of this phase is to integrate the cleaned batting and salary datasets into a single analytical dataset for subsequent analysis and modeling tasks.

This process includes reviewing the integration keys, preparing both datasets at player-season level, merging them using `player` and `year`, and validating the consistency of the resulting dataset.

Notebook: 03_Data_Integration

Author: Ronald Báez


In [28]:
# Import libraries
import pandas as pd
import numpy as np

In [29]:
# Load cleaned datasets
batting_clean = pd.read_csv("../01_data/02_processed_data/batting_clean.csv")
salary_clean = pd.read_csv("../01_data/02_processed_data/salary_clean.csv")


# Display data dimensions
print("Batting dataset shape:", batting_clean.shape)
print("Salary dataset shape:", salary_clean.shape)

Batting dataset shape: (4502, 36)
Salary dataset shape: (13953, 4)


In [30]:
# Display column names
print("Batting columns:")
print(batting_clean.columns.tolist())

print("\nSalary columns:")
print(salary_clean.columns.tolist())

Batting columns:
['rk', 'player', 'age', 'team', 'lg', 'war', 'g', 'pa', 'ab', 'r', 'h', '2b', '3b', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'ba', 'obp', 'slg', 'ops', 'ops_plus', 'roba', 'rbat_plus', 'tb', 'gidp', 'hbp', 'sh', 'sf', 'ibb', 'pos', 'awards', 'year', 'rownum']

Salary columns:
['year', 'team', 'player', 'salary']


###  Review Integration Keys

Before merging both datasets, it is necessary to review the variables that will be used as integration keys. In this project, the datasets will be merged using `player` and `year`, since there is no unique player identifier available.

In [31]:
# Review integration keys and year ranges
print("Batting key columns:")
print(batting_clean[["player", "year"]].dtypes)

print("\nSalary key columns:")
print(salary_clean[["player", "year"]].dtypes)

print("\nBatting years:", batting_clean["year"].min(), "-", batting_clean["year"].max())
print("Salary years:", salary_clean["year"].min(), "-", salary_clean["year"].max())

# Identify common years
common_years = sorted(set(batting_clean["year"]).intersection(set(salary_clean["year"])))

print("\nCommon years:", common_years)
print("Number of common years:", len(common_years))

# Check exact duplicates
print("\nExact duplicates in batting dataset:", batting_clean.duplicated().sum())
print("Exact duplicates in salary dataset:", salary_clean.duplicated().sum())

# Check duplicated player-year combinations
print("\nDuplicated player-year records in batting dataset:",
      batting_clean.duplicated(subset=["player", "year"]).sum())

print("Duplicated player-year records in salary dataset:",
      salary_clean.duplicated(subset=["player", "year"]).sum())

Batting key columns:
player    object
year       int64
dtype: object

Salary key columns:
player    object
year       int64
dtype: object

Batting years: 2015 - 2024
Salary years: 2011 - 2024

Common years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Number of common years: 10

Exact duplicates in batting dataset: 0
Exact duplicates in salary dataset: 0

Duplicated player-year records in batting dataset: 265
Duplicated player-year records in salary dataset: 60


In [32]:
# Inspect duplicated player-year records in batting dataset
batting_clean[
    batting_clean.duplicated(subset=["player", "year"], keep=False)
].sort_values(["player", "year", "team"]).head(20)

,rk,player,age,team,lg,war,g,pa,ab,r,...,tb,gidp,hbp,sh,sf,ibb,pos,awards,year,rownum
3,193.0,aaron hill,34.0,2TM,2LG,1.5,125.0,429.0,378.0,48.0,...,143.0,6.0,3.0,0.0,4.0,2.0,54H/D,no_award,2016,1
828,193.0,aaron hill,34.0,BOS,AL,-0.5,47.0,137.0,124.0,14.0,...,36.0,1.0,1.0,0.0,0.0,2.0,5H/4D,no_award,2016,1
2472,193.0,aaron hill,34.0,MIL,NL,1.9,78.0,292.0,254.0,34.0,...,107.0,5.0,2.0,0.0,4.0,0.0,54/H,no_award,2016,1
4,301.0,abraham almonte,26.0,2TM,2LG,0.8,82.0,258.0,232.0,36.0,...,95.0,5.0,0.0,3.0,2.0,0.0,8H/79D,no_award,2015,1
1369,301.0,abraham almonte,26.0,CLE,AL,1.1,51.0,196.0,178.0,30.0,...,81.0,4.0,0.0,0.0,2.0,0.0,8/H,no_award,2015,1
10,273.0,adam rosales,34.0,2TM,2LG,-0.7,105.0,312.0,289.0,25.0,...,102.0,6.0,4.0,4.0,4.0,1.0,65H4/37D,no_award,2017,1
3027,273.0,adam rosales,34.0,OAK,AL,0.0,71.0,223.0,205.0,15.0,...,71.0,2.0,2.0,3.0,3.0,0.0,6/4H57D,no_award,2017,1
11,251.0,adeiny hechavarria,28.0,2TM,2LG,0.3,97.0,348.0,330.0,37.0,...,134.0,7.0,1.0,2.0,2.0,1.0,6/H,no_award,2017,1
3962,251.0,adeiny hechavarria,28.0,TBR,AL,0.1,77.0,281.0,265.0,29.0,...,109.0,6.0,1.0,1.0,2.0,1.0,6/H,no_award,2017,1
13,268.0,adeiny hechavarria,29.0,2TM,AL,0.6,79.0,274.0,253.0,32.0,...,85.0,4.0,0.0,2.0,6.0,1.0,6/5H,no_award,2018,1


In [33]:
# Inspect duplicated player-year records in salary dataset
salary_clean[
    salary_clean.duplicated(subset=["player", "year"], keep=False)
].sort_values(["player", "year", "team"]).head(20)

,year,team,player,salary
6049,2022,COL,austin wynns,576779
7090,2022,SF,austin wynns,457674
6038,2022,COL,brent suter,3000000
6797,2022,MIL,brent suter,2700000
12544,2017,CHC,brett anderson,3500000
11865,2017,TOR,brett anderson,99382
5323,2023,CHW,carlos perez,267099
5444,2023,OAK,carlos perez,740000
6916,2022,CIN,chase anderson,142302
6051,2022,COL,chase anderson,549682


###  Prepare Batting Dataset at Player-Season Level

The batting dataset may contain more than one record for the same player and season when a player appeared for multiple teams during the same year. In these cases, aggregated team records such as `2TM`, `3TM`, or `4TM` are prioritized, since they represent the player's total season performance.

This step ensures that the batting dataset contains only one record per `player` and `year` before integration.

In [34]:
# Create a copy of the batting dataset
batting_player_year = batting_clean.copy()

# Identify aggregated multi-team records such as 2TM, 3TM or 4TM
batting_player_year["multi_team_flag"] = batting_player_year["team"].astype(str).str.match(r"^\d+TM$")

# Prioritize aggregated multi-team records and keep one row per player-year
batting_player_year = (
    batting_player_year
    .sort_values(
        by=["player", "year", "multi_team_flag", "pa"],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=["player", "year"], keep="first")
    .drop(columns=["multi_team_flag"])
    .reset_index(drop=True)
)

# Validate resulting dataset
print("Original batting dataset shape:", batting_clean.shape)
print("Player-year batting dataset shape:", batting_player_year.shape)
print("Remaining duplicated player-year records:",
      batting_player_year.duplicated(subset=["player", "year"]).sum())

Original batting dataset shape: (4502, 36)
Player-year batting dataset shape: (4237, 36)
Remaining duplicated player-year records: 0


In [35]:
# Count retained multi-team aggregated records
multi_team_records = batting_player_year[
    batting_player_year["team"].astype(str).str.match(r"^\d+TM$")
]

print("Retained multi-team aggregated records:", multi_team_records.shape[0])

Retained multi-team aggregated records: 413


###  Prepare Salary Dataset at Player-Season Level

The salary dataset may contain more than one record for the same player and year, usually due to team changes or multiple salary records within the same season.

To ensure consistency with the batting dataset, salaries are aggregated at player-season level by summing the salary values for each `player` and `year` combination.

In [36]:
# Aggregate salary dataset at player-season level
salary_player_year = (
    salary_clean
    .groupby(["player", "year"], as_index=False)
    .agg({
        "salary": "sum"
    })
)

# Validate resulting dataset
print("Original salary dataset shape:", salary_clean.shape)
print("Player-year salary dataset shape:", salary_player_year.shape)
print("Remaining duplicated player-year records:",
      salary_player_year.duplicated(subset=["player", "year"]).sum())

Original salary dataset shape: (13953, 4)
Player-year salary dataset shape: (13893, 3)
Remaining duplicated player-year records: 0


In [37]:
# Review salary distribution after aggregation
salary_player_year["salary"].describe()

count    1.389300e+04
mean     3.316198e+06
std      5.409583e+06
min      2.984000e+03
25%      4.800000e+05
50%      7.407410e+05
75%      3.750000e+06
max      5.500000e+07
Name: salary, dtype: float64

###  Merge Batting and Salary Datasets

Once both datasets have been prepared at player-season level, they can be merged using `player` and `year` as integration keys.

An inner join is used because the objective of the project is to analyze players with both offensive performance data and salary information available.

In [38]:
# Merge batting and salary datasets
mlb_hitters_integrated = batting_player_year.merge(
    salary_player_year,
    on=["player", "year"],
    how="inner"
)

# Validate merged dataset
print("Batting player-year shape:", batting_player_year.shape)
print("Salary player-year shape:", salary_player_year.shape)
print("Integrated dataset shape:", mlb_hitters_integrated.shape)

Batting player-year shape: (4237, 36)
Salary player-year shape: (13893, 3)
Integrated dataset shape: (3684, 37)


In [39]:
# Check duplicated player-year records after merge
integrated_duplicates = mlb_hitters_integrated.duplicated(subset=["player", "year"]).sum()

print("Duplicated player-year records in integrated dataset:", integrated_duplicates)

Duplicated player-year records in integrated dataset: 0


In [40]:
# Review final year range
print("Integrated dataset years:",
      mlb_hitters_integrated["year"].min(),
      "-",
      mlb_hitters_integrated["year"].max())

print("Number of seasons:",
      mlb_hitters_integrated["year"].nunique())

print("Number of unique players:",
      mlb_hitters_integrated["player"].nunique())

Integrated dataset years: 2015 - 2024
Number of seasons: 10
Number of unique players: 1033


###  Validate Integrated Dataset

Basic validation checks are performed to confirm that the integrated dataset has the expected structure, contains no duplicated player-season records, and covers the correct time period.

In [41]:
# Basic validation of the integrated dataset
print("Integrated dataset shape:", mlb_hitters_integrated.shape)
print("Unique players:", mlb_hitters_integrated["player"].nunique())
print("Year range:", mlb_hitters_integrated["year"].min(), "-", mlb_hitters_integrated["year"].max())
print("Duplicated player-year records:",
      mlb_hitters_integrated.duplicated(subset=["player", "year"]).sum())

Integrated dataset shape: (3684, 37)
Unique players: 1033
Year range: 2015 - 2024
Duplicated player-year records: 0


In [42]:
# Check missing values
mlb_hitters_integrated.isnull().sum().sort_values(ascending=False).head(10)

rk        0
player    0
age       0
team      0
lg        0
war       0
g         0
pa        0
ab        0
r         0
dtype: int64

In [43]:
# Records by year
mlb_hitters_integrated["year"].value_counts().sort_index()

year
2015    337
2016    343
2017    409
2018    405
2019    407
2020    263
2021    388
2022    379
2023    374
2024    379
Name: count, dtype: int64

###  Export Integrated Dataset

The final integrated dataset is exported to the processed data directory. This dataset will be used in the next phases for exploratory data analysis, feature engineering, and modeling.

In [44]:
# Export integrated dataset
mlb_hitters_integrated.to_csv(
    "../01_data/03_final_data/mlb_hitters_integrated.csv",
    index=False
)

print("Integrated dataset exported successfully.")

Integrated dataset exported successfully.


###  Conclusion

In this phase, the cleaned batting and salary datasets were integrated into a single analytical dataset at player-season level.

The integration was performed using `player` and `year` as key variables. Before merging, both datasets were prepared to ensure one record per player and season, avoiding duplication issues during the integration process.

The final integrated dataset was validated and exported for use in the next phases of the project, including exploratory data analysis, feature engineering, and predictive modeling.